In [13]:
import pandas as pd
import numpy as np
%load_ext autoreload
%autoreload 2            

import os, sys
sys.path.append(os.path.abspath(".."))   # ".." = thư mục cha của notebooks/ = algo-ml

from src import data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
df = pd.read_csv("../data/raw/EURUSD_H1.csv",index_col="timestamp",parse_dates=True)
df

,open,high,low,close,volume
timestamp,,,,,
2025-01-01 22:00:00+00:00,1.03503,1.03577,1.03483,1.03550,1443.22
2025-01-01 23:00:00+00:00,1.03550,1.03563,1.03462,1.03507,4447.37
2025-01-02 00:00:00+00:00,1.03508,1.03564,1.03472,1.03520,3681.37
2025-01-02 01:00:00+00:00,1.03521,1.03606,1.03442,1.03578,5012.45
2025-01-02 02:00:00+00:00,1.03577,1.03712,1.03577,1.03709,2791.47
...,...,...,...,...,...
2025-12-31 17:00:00+00:00,1.17346,1.17390,1.17312,1.17358,3662.76
2025-12-31 18:00:00+00:00,1.17357,1.17386,1.17302,1.17357,3944.61
2025-12-31 19:00:00+00:00,1.17356,1.17432,1.17356,1.17407,3321.68


In [15]:
print(df.index.duplicated().sum()) #Không có duplicate

0


In [16]:
df['time_diff'] = df.index.diff()

print(df['time_diff'].value_counts()) #Missing candle trong ngày 25 giáng sinh, thiếu 15 candles trùng với 15 candle cần tìm

time_diff
0 days 01:00:00    6170
2 days 01:00:00      50
2 days 00:00:00       1
2 days 02:00:00       1
0 days 02:00:00       1
0 days 15:00:00       1
Name: count, dtype: int64


In [17]:
idx = df.index
if idx.tz is None:                      # index chưa gắn múi giờ thì gắn UTC
    idx = idx.tz_localize('UTC')

# 1) Lịch đầy đủ: mỗi giờ 1 nến, từ nến đầu đến nến cuối
full = pd.date_range(start=idx[0], end=idx[-1], freq='h')

# 2) Giờ có trong lịch nhưng không có trong dữ liệu (vẫn còn lẫn cuối tuần)
missing = full.difference(idx)

# 3) Đổi sang giờ New York: thị trường đóng từ thứ Sáu 17h đến Chủ nhật 17h
ny = missing.tz_convert('America/New_York')
weekend = ((ny.day_of_week == 4) & (ny.hour >= 17)) \
        | (ny.day_of_week == 5) \
        | ((ny.day_of_week == 6) & (ny.hour < 17))
missing_trading = missing[~weekend]     # ~ nghĩa là "không phải"
print(len(missing_trading))
print(missing_trading)

15
DatetimeIndex(['2025-12-01 09:00:00+00:00', '2025-12-25 08:00:00+00:00',
               '2025-12-25 09:00:00+00:00', '2025-12-25 10:00:00+00:00',
               '2025-12-25 11:00:00+00:00', '2025-12-25 12:00:00+00:00',
               '2025-12-25 13:00:00+00:00', '2025-12-25 14:00:00+00:00',
               '2025-12-25 15:00:00+00:00', '2025-12-25 16:00:00+00:00',
               '2025-12-25 17:00:00+00:00', '2025-12-25 18:00:00+00:00',
               '2025-12-25 19:00:00+00:00', '2025-12-25 20:00:00+00:00',
               '2025-12-25 21:00:00+00:00'],
              dtype='datetime64[us, UTC]', freq=None)


In [18]:
# Điều kiện giá <= 0
invalid_price = ((df[['open', 'high', 'low', 'close']] <= 0).any(axis=1))
# Điều kiện OHLC không hợp lệ
invalid_ohlc = (
    (df['high'] < df['open']) |
    (df['high'] < df['close']) |
    (df['low'] > df['open']) |
    (df['low'] > df['close']) |
    (df['high'] < df['low'])
)
invalid_candles = invalid_price | invalid_ohlc
bad_rows = df[invalid_candles]

print(f"Invalid candles: {len(bad_rows)}")
print(bad_rows)

Invalid candles: 0
Empty DataFrame
Columns: [open, high, low, close, volume, time_diff]
Index: []


In [19]:
df["ret"] = df["close"].pct_change()
df["range"] = df["high"] - df["low"]

In [20]:
df.describe()

,open,high,low,close,volume,time_diff,ret,range
count,6225.000000,6225.000000,6225.000000,6225.000000,6225.000000,6224,6224.000000,6225.000000
mean,1.130421,1.131207,1.129683,1.130448,5952.833774,0 days 01:24:12.377892,0.000021,0.001525
std,0.048842,0.048771,0.048896,0.048825,5181.900837,0 days 04:22:22.187137,0.001013,0.001145
min,1.019310,1.020400,1.017720,1.019320,9.450000,0 days 01:00:00,-0.010560,0.000200
25%,1.092330,1.093570,1.091540,1.092490,2568.710000,0 days 01:00:00,-0.000427,0.000830
50%,1.153710,1.154330,1.152940,1.153710,4490.890000,0 days 01:00:00,0.000009,0.001230
75%,1.166640,1.167330,1.165970,1.166630,7535.150000,0 days 01:00:00,0.000432,0.001840
max,1.187110,1.191860,1.186550,1.187090,54210.770000,2 days 02:00:00,0.012328,0.016690


In [21]:
vol = df['ret'].rolling(24).std().shift(1)   # dao động của 24 nến H1 trước đó
z = df['ret'].abs() / vol                     # nến này lớn gấp mấy lần bình thường
for k in [5, 8, 12]:
    print(k, (z > k).sum())

5 30
8 7
12 1


In [22]:
flag = df[z > 8].copy()
flag['dow'] = flag.index.day_of_week   # 0 = thứ Hai ... 6 = Chủ nhật
flag['hour'] = flag.index.hour       # giờ UTC
print(flag)

                              open     high      low    close     volume  \
timestamp                                                                  
2025-01-06 11:00:00+00:00  1.03349  1.04325  1.03287  1.04217  27467.570   
2025-01-10 13:00:00+00:00  1.03054  1.03118  1.02127  1.02563  45832.790   
2025-01-20 13:00:00+00:00  1.03200  1.03999  1.03077  1.03980  27188.560   
2025-05-12 07:00:00+00:00  1.12179  1.12190  1.10977  1.11048  33297.160   
2025-08-01 12:00:00+00:00  1.14045  1.15584  1.13915  1.15451  26416.170   
2025-08-22 14:00:00+00:00  1.15996  1.17199  1.15996  1.17180  28427.458   
2025-09-02 07:00:00+00:00  1.17031  1.17064  1.16426  1.16458  12511.830   

                                time_diff       ret    range  dow  hour  
timestamp                                                                
2025-01-06 11:00:00+00:00 0 days 01:00:00  0.008389  0.01038    0    11  
2025-01-10 13:00:00+00:00 0 days 01:00:00 -0.004774  0.00991    4    13  
2025-01-20 13:00:00

In [23]:
data.quality_report(df,8)

There are: 15 missing candles
DatetimeIndex(['2025-12-01 09:00:00+00:00', '2025-12-25 08:00:00+00:00',
               '2025-12-25 09:00:00+00:00', '2025-12-25 10:00:00+00:00',
               '2025-12-25 11:00:00+00:00', '2025-12-25 12:00:00+00:00',
               '2025-12-25 13:00:00+00:00', '2025-12-25 14:00:00+00:00',
               '2025-12-25 15:00:00+00:00', '2025-12-25 16:00:00+00:00',
               '2025-12-25 17:00:00+00:00', '2025-12-25 18:00:00+00:00',
               '2025-12-25 19:00:00+00:00', '2025-12-25 20:00:00+00:00',
               '2025-12-25 21:00:00+00:00'],
              dtype='datetime64[us, UTC]', freq=None)
There are no duplicated candles
Invalid price: 0
Invalid OHLC: 0
Invalid candles: 0
Empty DataFrame
Columns: [open, high, low, close, volume, time_diff, ret, range]
Index: []
                              open     high      low    close     volume  \
timestamp                                                                  
2025-01-06 11:00:00+00:00  1.0334

rows          6225
duplicated       0
missing         15
bad_price        0
bad_ohlc         0
spikes           7
dtype: int64

In [24]:
cols = ['open', 'high', 'low', 'close', 'volume']   # bỏ các cột tự thêm (time_diff, ret, range)
raw = df[cols]

csv_path = '../data/raw/EURUSD_H1.csv'
pq_path  = '../data/raw/EURUSD_H1.parquet'

raw.to_parquet(pq_path)
back = pd.read_parquet(pq_path)
print(back.index.dtype, back.equals(raw))           # kiểu index có giữ nguyên không, có khớp từng giá trị không

print(os.path.getsize(csv_path) / 1024, 'KB CSV')
print(os.path.getsize(pq_path) / 1024, 'KB parquet')

plain = pd.read_csv(csv_path)                       # bài sinh đôi cho câu c
print(plain.dtypes)

datetime64[us, UTC] True
404.666015625 KB CSV
275.8603515625 KB parquet
timestamp        str
open         float64
high         float64
low          float64
close        float64
volume       float64
dtype: object
